# Capitalization embeddings smoke test

Run this from the repo root in Colab or a local Jupyter environment. This notebook checks that the uncased tokenizer still emits normal BERT IDs while a parallel `capitalization_ids` tensor preserves first-cap and all-caps information.

In [ ]:
# In Colab, uncomment and adjust if needed:
# %cd /content/CapitalizationEmbeddings
%pip install -q -e . -r requirements-colab.txt

In [ ]:
from pathlib import Path

if not Path("pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the CapitalizationEmbeddings repo root.")

In [ ]:
from transformers import AutoTokenizer

from capitalization_embeddings import tokenize_with_capitalization

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

examples = [
    "Tom met tom and TOM near iPhone HQ.",
    "NASA hired Alice in New York.",
]

batch = tokenize_with_capitalization(
    tokenizer,
    examples,
    padding=True,
    truncation=True,
    max_length=32,
)

for text, input_ids, capitalization_ids in zip(
    examples,
    batch["input_ids"],
    batch["capitalization_ids"],
):
    print("\n" + text)
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    for token, cap_id in zip(tokens, capitalization_ids):
        if token != tokenizer.pad_token:
            print(f"{token:>12}  cap_id={cap_id}")

In [ ]:
import torch

from capitalization_embeddings import (
    CapitalizedBertForMaskedLM,
    DataCollatorForCapitalizedLanguageModeling,
)

features = [
    tokenize_with_capitalization(
        tokenizer,
        example,
        truncation=True,
        max_length=32,
    )
    for example in examples
]

collator = DataCollatorForCapitalizedLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.30,
)

torch.manual_seed(0)
mlm_batch = collator(features)

model = CapitalizedBertForMaskedLM.from_uncased_pretrained("bert-base-uncased")
model.eval()

with torch.no_grad():
    outputs = model(**mlm_batch)

print("loss:", float(outputs.loss))
print("token logits:", tuple(outputs.logits.shape))
print("capitalization logits:", tuple(outputs.capitalization_logits.shape))